# Diabetic Retinopathy Screening - Training Notebook

This notebook is a thin wrapper around the scripts in `model/` so you can run
the pipeline interactively on Colab with a free GPU.

Steps:
1. Mount Google Drive or upload the APTOS 2019 dataset
2. Install dependencies
3. Train
4. Evaluate
5. Try Grad-CAM on a few example images

In [ ]:
!pip install -q torch torchvision opencv-python scikit-learn matplotlib pytorch-grad-cam pandas tqdm

In [ ]:
# Point this at wherever you've placed train.csv and train_images/
# e.g. after downloading from Kaggle and unzipping into /content/data
DATA_DIR = "/content/data"

In [ ]:
%cd model
!python train.py --data_dir {DATA_DIR} --epochs 15 --batch_size 32

In [ ]:
!python evaluate.py --data_dir {DATA_DIR} --checkpoint best_model.pth

In [ ]:
from gradcam import load_model_for_cam, generate_gradcam
import torch, matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model_for_cam('best_model.pth', device)

# swap in a path to one of your test images
example_image_path = f'{DATA_DIR}/train_images/EXAMPLE_ID.png'
overlay, pred_class, probs = generate_gradcam(model, example_image_path, device)

class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
plt.imshow(overlay)
plt.title(f'Predicted: {class_names[pred_class]} ({probs[pred_class]:.1%})')
plt.axis('off')
plt.show()